# 第10章 性能評価

## 10.2 評価指標を用いた自動評価

### 10.2.4 多肢選択式質問応答タスクによる自動評価

#### 環境準備

In [1]:
!pip install transformers[torch,sentencepiece] bitsandbytes 'datasets<4.0.0'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 41.9 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [2]:
from transformers.trainer_utils import set_seed
set_seed(42)

#### データセットの準備

In [3]:
from datasets import load_dataset
train_dataset = load_dataset(
    "llm-book/JGLUE", name="JCommonsenseQA", split="train"
)
val_dataset = load_dataset(
    "llm-book/JGLUE", name="JCommonsenseQA", split="validation"
)
print(val_dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

JGLUE.py:   0%|          | 0.00/13.9k [00:00<?, ?B/s]

preprocess_marc_ja.py:   0%|          | 0.00/9.03k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['q_id', 'question', 'choice0', 'choice1', 'choice2', 'choice3', 'choice4', 'label'],
    num_rows: 1119
})


In [ ]:
print(train_dataset[0])

{'q_id': 0, 'question': '主に子ども向けのもので、イラストのついた物語が書かれているものはどれ？', 'choice0': '世界', 'choice1': '写真集', 'choice2': '絵本', 'choice3': '論文', 'choice4': '図鑑', 'label': 2}


#### データの前処理

In [5]:
from pprint import pprint

def convert_data_format(data: dict[str, str]) -> dict[str, str]:
    """選択肢の中から質問に数字で回答する形式にデータを変換する"""
    data["input"] = (
        f"質問：{data['question']}\n"
        f"選択肢：0.{data['choice0']},1.{data['choice1']},"
        f"2.{data['choice2']},3.{data['choice3']},"
        f"4.{data['choice4']}"
    )
    data["output"] = data["label"]
    return data

In [6]:
# 訓練セットをシャッフルする
train_dataset = train_dataset.shuffle()
# 訓練セットの前処理をする
train_dataset = train_dataset.map(convert_data_format)
# 4つのfew-shot事例を取得する
few_shots = list(train_dataset)[:4]
# 検証セットの前処理をする
val_dataset = val_dataset.map(convert_data_format)
pprint(list(val_dataset)[0])

Map:   0%|          | 0/8939 [00:00<?, ? examples/s]

Map:   0%|          | 0/1119 [00:00<?, ? examples/s]

{'choice0': '掲示板',
 'choice1': 'パソコン',
 'choice2': 'マザーボード',
 'choice3': 'ハードディスク',
 'choice4': 'まな板',
 'input': '質問：電子機器で使用される最も主要な電子回路基板の事をなんと言う？\n'
          '選択肢：0.掲示板,1.パソコン,2.マザーボード,3.ハードディスク,4.まな板',
 'label': 2,
 'output': 2,
 'q_id': 8939,
 'question': '電子機器で使用される最も主要な電子回路基板の事をなんと言う？'}


In [7]:
print(few_shots[0])

{'q_id': 8530, 'question': '4輪でハンドルで操作し、ガソリンや電気で動く乗り物は？', 'choice0': '自動車', 'choice1': '自転車', 'choice2': 'イヤホン', 'choice3': '飛行機', 'choice4': 'ライブ', 'label': 0, 'input': '質問：4輪でハンドルで操作し、ガソリンや電気で動く乗り物は？\n選択肢：0.自動車,1.自転車,2.イヤホン,3.飛行機,4.ライブ', 'output': 0}


#### プロンプトテンプレートの作成

In [8]:
def create_prompt_template(
    instruction: str,
    few_shots: list[dict[str, str]] | None = None
) -> str:
    """プロンプトテンプレートを作成する"""
    prompt_template = (
        "以下は、タスクを説明する指示と、"
        "文脈のある入力の組み合わせです。"
        "要求を適切に満たす応答を書きなさい。\n\n"
    )
    prompt_template += f"### 指示\n{instruction}\n\n"
    if few_shots is not None:
        for few_shot in few_shots:
            prompt_template += f"### 入力:\n{few_shot["input"]}\n\n"
            prompt_template += f"### 応答:\n{few_shot["output"]}\n\n"
    prompt_template += "### 入力:\n{input}\n\n"
    prompt_template += "### 応答:\n"
    return prompt_template

In [9]:
# 指示文を指定してプロンプトテンプレートを作成する
instruction = """
質問と解答の選択肢を入力として受け取り、選択肢から回答を選択してください。
なお、回答は選択肢の番号（例：0）でするものとします。 
回答となる数値をint型で返し、他には何も含めないことを厳守してください。
"""
prompt_template = create_prompt_template(instruction, few_shots)
print(prompt_template)

以下は、タスクを説明する指示と、文脈のある入力の組み合わせです。要求を適切に満たす応答を書きなさい。

### 指示

質問と解答の選択肢を入力として受け取り、選択肢から回答を選択してください。
なお、回答は選択肢の番号（例：0）でするものとします。 
回答となる数値をint型で返し、他には何も含めないことを厳守してください。


### 入力:
質問：4輪でハンドルで操作し、ガソリンや電気で動く乗り物は？
選択肢：0.自動車,1.自転車,2.イヤホン,3.飛行機,4.ライブ

### 応答:
0

### 入力:
質問：夏になったら着たくなるものは？
選択肢：0.誘い水,1.水着,2.化粧水,3.水すまし,4.水筒

### 応答:
1

### 入力:
質問：声や楽器を使った芸術は？
選択肢：0.猫,1.音楽,2.展覧会,3.歌,4.スズメ

### 応答:
1

### 入力:
質問：売る物のことを何と言うか？
選択肢：0.自転車,1.鍋,2.車,3.飲み物,4.商品

### 応答:
4

### 入力:
{input}

### 応答:



#### パイプラインの作成

In [11]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    pipeline
)

model_name = "tokyotech-llm/Swallow-7b-instruct-hf"
tokenizer = AutoTokenizer.from_pretrained(model_name)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config,
    use_cache=False,
    device_map="auto",
)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

In [12]:
# テキスト生成用のパラメータを指定する
generation_config = {
    "max_new_tokens": 1, # 生成する最大トークン数
    "top_p": 1.0, # top-pサンプリング
    "repetition_penalty": 1.0, # 繰り返しペナルティ
}
# pipelineを作成する
text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    **generation_config
)

Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'max_new_tokens', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


#### 質問の回答を生成

In [13]:
from datasets import Dataset
from tqdm import tqdm
from transformers import TextGenerationPipeline

def generate_answers(
    text_generation_pipeline: TextGenerationPipeline,
    dataset: Dataset,
    prompt_template: str,
) -> list[dict[str, str]]:
    """プロンプトを使って質問の回答を生成する"""
    results = []
    for data in tqdm(dataset):
        # プロンプトテンプレートの{input}を質問テキストに置換する
        prompt = prompt_template.format(input=data["input"])
        # 質問の回答を生成する
        output = text_generation_pipeline(prompt)
        # プロンプト部分を削除して予測部分のみにする
        generated_text = output[0]["generated_text"].replace(
            prompt, ""
        )
        # 複数行出力された場合に、最初の行だけを抽出して回答部分のみにする
        pred_label = generated_text.split("\n")[0].strip()
        results.append(
            {
                "input": data["input"],
                "true_label": data["output"],
                "pred_label": pred_label,
            }
        )
    return results

In [14]:
# 検証セットに対して質問の回答を生成する
results1 = generate_answers(
    text_generation_pipeline, val_dataset, prompt_template
)
pprint(results1[:3])

Both `max_new_tokens` (=1) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

[{'input': '質問：電子機器で使用される最も主要な電子回路基板の事をなんと言う？\n'
           '選択肢：0.掲示板,1.パソコン,2.マザーボード,3.ハードディスク,4.まな板',
  'pred_label': '4',
  'true_label': 2},
 {'input': '質問：田んぼが広がる風景を何という？\n選択肢：0.畑,1.海,2.田園,3.地方,4.牧場',
  'pred_label': '4',
  'true_label': 2},
 {'input': '質問：しゃがんだりする様を何という？\n選択肢：0.腰を下す,1.座る,2.仮眠を取る,3.寝る,4.起きる',
  'pred_label': '4',
  'true_label': 0}]


In [15]:
from google.colab import drive

# Googleドライブを"drive"ディレクトリ以下にマウントする
drive.mount("drive")

Mounted at drive


In [16]:
import json
from pathlib import Path

def write_jsonl(path: str, items: list[dict]) -> None:
    """JSON Lines形式で出力する"""
    # 保存先のフォルダが存在しない場合は作成する
    Path(path).parent.mkdir(parents=True, exist_ok=True)

    # ファイルに書き込む
    with open(path, "w") as f:
        for item in items:
            print(json.dumps(item, ensure_ascii=False), file=f)

# 出力結果をresult1.jsonlというファイルに書き込む
output_path = "./drive/MyDrive/llm-book/eval/jcommonsenseqa/results1.jsonl"
write_jsonl(output_path, results1)

#### 完全一致率の算出

In [17]:
def calc_exact_match_ratio(
    trues: list[str],
    preds: list[str]
) -> float:
    """完全一致率を算出する"""
    # どちらかの事例がなければ0
    if len(trues) == 0 or len(preds) == 0:
        return 0
    # 正解テキストと予測テキストが一致していれば1, そうでなければ0
    num_exact_match = sum(
        1 if t == p else 0 for t, p in zip(trues, preds)
    )
    return num_exact_match / len(trues)

In [18]:
# 完全一致率を算出する
true_labels1 = [str(r["true_label"]) for r in results1]
pred_labels1 = [str(r["pred_label"]) for r in results1]
score = calc_exact_match_ratio(true_labels1, pred_labels1)
print("完全一致率: ", score)

完全一致率:  0.1680071492403932


In [19]:
pprint(results1[:10])

[{'input': '質問：電子機器で使用される最も主要な電子回路基板の事をなんと言う？\n'
           '選択肢：0.掲示板,1.パソコン,2.マザーボード,3.ハードディスク,4.まな板',
  'pred_label': '4',
  'true_label': 2},
 {'input': '質問：田んぼが広がる風景を何という？\n選択肢：0.畑,1.海,2.田園,3.地方,4.牧場',
  'pred_label': '4',
  'true_label': 2},
 {'input': '質問：しゃがんだりする様を何という？\n選択肢：0.腰を下す,1.座る,2.仮眠を取る,3.寝る,4.起きる',
  'pred_label': '4',
  'true_label': 0},
 {'input': '質問：水を出すときに捻るものは？\n選択肢：0.蛇口,1.ハンドル,2.流し,3.釘,4.食器棚',
  'pred_label': '4',
  'true_label': 0},
 {'input': '質問：音楽にあわせて踊る場所は？\n選択肢：0.ディスコ,1.市長,2.イチゴ,3.ショッピングセンター,4.マンゴー',
  'pred_label': '4',
  'true_label': 0},
 {'input': '質問：地中にある一定の大きさの空間のこと？\n選択肢：0.麓,1.山頂,2.中腹,3.山腹,4.洞窟',
  'pred_label': '5',
  'true_label': 4},
 {'input': '質問：乗り物が関係する言葉はどれ？\n選択肢：0.外に出る,1.歩く,2.靴を脱ぐ,3.靴を履く,4.車から降りる',
  'pred_label': '5',
  'true_label': 4},
 {'input': '質問：持ち運び、使いまわりの良さに富んだパソコンをなんという？\n'
           '選択肢：0.コンピュータ,1.ロープ,2.ノートパソコン,3.携帯,4.軽食',
  'pred_label': '1',
  'true_label': 2},
 {'input': '質問：タバコを吸う事を何と言う？\n選択肢：0.電話,1.食事,2.喫煙,3.仏,4.飲酒',
  '

#### 文字回答形式の評価結果

選択肢ではなく文字列で回答する
```
### 入力:
質問：夏になったら着たくなるものは？
選択肢：誘い水,水着,化粧水,水すまし,水筒

### 応答:
水着
```

In [20]:
def convert_data_format2(data: dict[str, str]) -> dict[str, str]:
    """選択肢の中から質問に文字列で回答する形式にデータを変換する"""
    data["input"] = (
        f"質問：{data['question']}\n"
        f"選択肢：{data['choice0']},{data['choice1']},"
        f"{data['choice2']},{data['choice3']},"
        f"{data['choice4']}"
    )
    choice = f"choice{data['label']}"
    data["output"] = data[choice]
    return data


In [21]:
# 訓練セットの前処理をする
train_dataset = train_dataset.map(convert_data_format2)
# 四つのfew-shot事例を取得する
few_shots2 = list(train_dataset)[:4]
# 開発セットの前処理をする
val_dataset = val_dataset.map(convert_data_format2)
print(list(val_dataset)[0])

Map:   0%|          | 0/8939 [00:00<?, ? examples/s]

Map:   0%|          | 0/1119 [00:00<?, ? examples/s]

{'q_id': 8939, 'question': '電子機器で使用される最も主要な電子回路基板の事をなんと言う？', 'choice0': '掲示板', 'choice1': 'パソコン', 'choice2': 'マザーボード', 'choice3': 'ハードディスク', 'choice4': 'まな板', 'label': 2, 'input': '質問：電子機器で使用される最も主要な電子回路基板の事をなんと言う？\n選択肢：掲示板,パソコン,マザーボード,ハードディスク,まな板', 'output': 'マザーボード'}


In [22]:
# プロンプトテンプレートを作成する
instruction2 = """
質問と回答の選択肢を入力として受け取り、選択肢から回答を選択してください。
なお、回答以外には何も含めないことを厳守してください。
"""
prompt_template2 = create_prompt_template(instruction2, few_shots2)
print(prompt_template2)

以下は、タスクを説明する指示と、文脈のある入力の組み合わせです。要求を適切に満たす応答を書きなさい。

### 指示

質問と回答の選択肢を入力として受け取り、選択肢から回答を選択してください。
なお、回答以外には何も含めないことを厳守してください。


### 入力:
質問：4輪でハンドルで操作し、ガソリンや電気で動く乗り物は？
選択肢：自動車,自転車,イヤホン,飛行機,ライブ

### 応答:
自動車

### 入力:
質問：夏になったら着たくなるものは？
選択肢：誘い水,水着,化粧水,水すまし,水筒

### 応答:
水着

### 入力:
質問：声や楽器を使った芸術は？
選択肢：猫,音楽,展覧会,歌,スズメ

### 応答:
音楽

### 入力:
質問：売る物のことを何と言うか？
選択肢：自転車,鍋,車,飲み物,商品

### 応答:
商品

### 入力:
{input}

### 応答:



In [23]:
# テキスト生成用のパラメータを指定する
generation_config2 = {
    "max_new_tokens": 10,
    "top_p": 1.0,
    "repetition_penalty": 1.0,
}
# pipelineを作成する
text_generation_pipeline2 = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    **generation_config2
)

In [26]:
# 検証セットに対して質問の回答を生成する
results2 = generate_answers(
    text_generation_pipeline2, val_dataset, prompt_template2
)
pprint(results2[:3])
output_path = "./drive/MyDrive/llm_book/eval/jcommonsenseqa/results2.jsonl"
write_jsonl(output_path, results2)

Both `max_new_tokens` (=10) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Both `max_new_tokens` (=10) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation

[{'input': '質問：電子機器で使用される最も主要な電子回路基板の事をなんと言う？\n選択肢：掲示板,パソコン,マザーボード,ハードディスク,まな板',
  'pred_label': '## ## ## ## # ## ## ## ## #',
  'true_label': 'マザーボード'},
 {'input': '質問：田んぼが広がる風景を何という？\n選択肢：畑,海,田園,地方,牧場',
  'pred_label': '▁product ## ## ## ## ## # ## # ##',
  'true_label': '田園'},
 {'input': '質問：しゃがんだりする様を何という？\n選択肢：腰を下す,座る,仮眠を取る,寝る,起きる',
  'pred_label': '# ## # # d # d ## d ##',
  'true_label': '腰を下す'}]


In [27]:
pprint(results2[:5])
output_path = "./drive/MyDrive/llm_book/eval/jcommonsenseqa/results2.jsonl"
write_jsonl(output_path, results2)

[{'input': '質問：電子機器で使用される最も主要な電子回路基板の事をなんと言う？\n選択肢：掲示板,パソコン,マザーボード,ハードディスク,まな板',
  'pred_label': '## ## ## ## # ## ## ## ## #',
  'true_label': 'マザーボード'},
 {'input': '質問：田んぼが広がる風景を何という？\n選択肢：畑,海,田園,地方,牧場',
  'pred_label': '▁product ## ## ## ## ## # ## # ##',
  'true_label': '田園'},
 {'input': '質問：しゃがんだりする様を何という？\n選択肢：腰を下す,座る,仮眠を取る,寝る,起きる',
  'pred_label': '# ## # # d # d ## d ##',
  'true_label': '腰を下す'},
 {'input': '質問：水を出すときに捻るものは？\n選択肢：蛇口,ハンドル,流し,釘,食器棚',
  'pred_label': '# ## # ## # # # # # ##',
  'true_label': '蛇口'},
 {'input': '質問：音楽にあわせて踊る場所は？\n選択肢：ディスコ,市長,イチゴ,ショッピングセンター,マンゴー',
  'pred_label': '## ## # <0x0A> : ## # ## # ##',
  'true_label': 'ディスコ'}]


In [ ]:
# 完全一致率を算出する
true_labels2 = [r["true_label"] for r in results2]
pred_labels2 = [r["pred_label"] for r in results2]
score2 = calc_exact_match_ratio(true_labels2, pred_labels2)
print("完全一致率:", score2)

完全一致率: 0.0


↑ 数字を認識する必要がなくなり回答精度向上